# Lab 1 — Tokens (EP01)

In the theory video we saw that an LLM reads and writes **tokens**, and that everything is billed in tokens.
In this lab we look at the real thing with **Gemma 4 26B** through OpenRouter.

**What you'll learn**
1. Call an LLM through an API and read `usage` (the number of tokens used)
2. See how text is cut into tokens — Thai compared with English
3. Count Gemma's real tokens, including the ones the chat template adds
4. Work out the cost, check the context window, and see an answer cut off by `max_tokens`

The sample text in this lab (the abstract) is **simulated data written for teaching**; every number and model answer comes from a real run.

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §1.1–1.2, §18.2 — CC BY-SA 4.0

## 0. Set up before you start

| What you need | Details |
|---|---|
| Python 3.10 or later | or use Google Colab, which already has it |
| OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (the whole episode costs less than 1 baht, a few cents at most) |
| Packages | `openai`, `python-dotenv`, `tiktoken`, `requests` (installed in the next cell) |

**Add your API key**
- **Colab:** 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file in the same folder as the notebook and add the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key directly in your code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv tiktoken requests

## 1. Calling Gemma through OpenRouter

**Why `google/gemma-4-26b-a4b-it`** — instruction-tuned (follows instructions), a MoE with ~4B active per token so it is fast and cheap,
it returns token probabilities (used in Lab 2) and accepts images (used in Lab 3)

**Not free**, but very cheap: all of EP01 costs less than one baht · the `:free` version has a single provider that doesn't return probabilities
and is rate-limited (tested 2026-11-08: 429 "rate-limited" 1 time in 3, and error 404 every time probabilities were requested)

In [ ]:
import os, requests, tiktoken
from openai import OpenAI

In [ ]:
def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

In [ ]:
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False}}

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "In the world of LLMs, what is a token? Answer in one sentence."}],
    extra_body=EXTRA,
)
print(resp.choices[0].message.content)
print(resp.usage.prompt_tokens, "input tokens ·", resp.usage.completion_tokens, "output tokens")

## 2. Seeing tokens with your own eyes

`tiktoken` is OpenAI's tokenizer (`o200k_base`), which runs right on your machine — **it is not Gemma's**,
but it shows how text is cut into pieces.

In [ ]:
enc = tiktoken.get_encoding("o200k_base")

def show_tokens(text):
    ids = enc.encode(text)
    pieces = [enc.decode([i]) for i in ids]
    print(len(ids), "token:", pieces)

In [ ]:
show_tokens("unhappiness")
show_tokens("Agents use tools to finish tasks.")
show_tokens("เอเจนต์ใช้เครื่องมือเพื่อทำงานให้เสร็จ")

## 3. Counting Gemma's real tokens

Gemma has its own tokenizer. The surest way is to let the API count — send the text with `max_tokens=1` and read `usage.prompt_tokens`

In [ ]:
def prompt_tokens(text):
    resp = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": text}],
        max_tokens=1, extra_body=EXTRA,
    )
    return resp.usage.prompt_tokens

In [ ]:
OVERHEAD = prompt_tokens("a") - 1
print("chat template adds", OVERHEAD, "token")

In [ ]:
for text in ["Agents use tools to finish tasks.", "เอเจนต์ใช้เครื่องมือเพื่อทำงานให้เสร็จ"]:
    gemma = prompt_tokens(text) - OVERHEAD
    print(f"tiktoken {len(enc.encode(text)):>2} · Gemma {gemma:>2} · {text}")

## 4. Cost and the context window

OpenRouter lists the price per token and the context window size of every model at `https://openrouter.ai/api/v1/models`

In [ ]:
models = requests.get("https://openrouter.ai/api/v1/models").json()["data"]
info = next(m for m in models if m["id"] == MODEL)
PRICE_IN = float(info["pricing"]["prompt"])
PRICE_OUT = float(info["pricing"]["completion"])
CONTEXT = info["context_length"]
print(f"{PRICE_IN * 1e6:.4g} $/million input tokens · {PRICE_OUT * 1e6:.4g} $/million output tokens · context {CONTEXT:,} token")

In [ ]:
u = resp.usage
cost = u.prompt_tokens * PRICE_IN + u.completion_tokens * PRICE_OUT
print(f"first question ≈ ${cost:.8f} · 1,000 questions ≈ ${cost * 1000:.4f}")

In [ ]:
ABSTRACT = """งานวิจัยนี้ศึกษาการใช้ transformer พยากรณ์อนุกรมเวลา เราเสนอวิธีแบ่งข้อมูลเป็นช่วงสั้นๆ
แล้วให้ attention เรียนรู้ความสัมพันธ์ระหว่างช่วง ผลการทดลองกับข้อมูลไฟฟ้าและการจราจร
แสดงว่าความคลาดเคลื่อนลดลงเมื่อเทียบกับ model เดิม และใช้เวลาฝึกน้อยลง"""
n = prompt_tokens(ABSTRACT)
print(f"abstract = {n} token · uses {n / CONTEXT:.3%} of the context · fits about {CONTEXT // n:,} abstracts")

## 5. A cut-off answer: `max_tokens` and `finish_reason`

In [ ]:
for limit in [20, 300]:
    r = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": "Explain what an LLM is in two sentences."}],
        max_tokens=limit, extra_body=EXTRA,
    )
    reason = r.choices[0].finish_reason
    print(f"max_tokens={limit} → finish_reason={reason!r} · {r.usage.completion_tokens} token")
    if reason == "length":
        print("  ✂️ cut off:", r.choices[0].message.content)

## Summary

| What we saw | Real numbers from this lab |
|---|---|
| The chat template adds tokens to every call | 13 tokens |
| Thai takes more tokens than English | English 7 · Thai 11 (Gemma) / 15 (tiktoken) |
| Different models count tokens differently | Gemma 11 vs tiktoken 15 for the same sentence |
| Output tokens cost more than input tokens | about 3× |
| `finish_reason="length"` = the answer was cut | at `max_tokens=20` |

**Exercise:** write 3 pairs of Thai and English questions with the same meaning, then measure how many times more tokens the Thai takes on average (use `prompt_tokens` and `OVERHEAD`)

▶ **Next: Lab 2 — Decoding** — look at Gemma's real probabilities, then write temperature, top-k, top-p and min-p yourself